In [1]:
import os
print(os.environ.get('TRAJECTORY_WORKDIR'))
print(os.environ.get('PROJECT_TAG'))


/workspace/23cc7603-1325-4953-8d3a-899ea46da553
None


In [2]:
import math

# === Sanity-check key geometry claims from the design ===

# Chassis
chassis_x, chassis_y, chassis_z = 70.0, 60.0, 300.0
auger_tube_outer_d = 25.0
auger_bore_d = auger_tube_outer_d + 0.4     # 25.4
bore_r = auger_bore_d/2                      # 12.7

# ---- Servo pocket "doesn't break bore" claim ----
servo_pocket_y = 16.3
servo_y_outer = chassis_y/2                  # 30
servo_y_inner = servo_y_outer - servo_pocket_y    # 13.7
wall_between_pocket_and_bore = servo_y_inner - bore_r
print(f"Servo pocket inner Y = {servo_y_inner:.2f}, bore radius = {bore_r:.2f}, residual wall = {wall_between_pocket_and_bore:.2f} mm")

# But the "horn_slot" is then explicitly cut THROUGH from pocket to bore.
# Let's check what y-extent the horn_slot box covers.
servo_horn_slot_w = 6.0   # X width
servo_horn_slot_h = 8.0   # Z height
# horn_slot center y = servo_y_inner/2 = 6.85 ; length in Y = servo_y_inner + bore_r/2
horn_slot_center_y = servo_y_inner/2
horn_slot_len_y = servo_y_inner + auger_bore_d/2
horn_slot_y_min = horn_slot_center_y - horn_slot_len_y/2
horn_slot_y_max = horn_slot_center_y + horn_slot_len_y/2
print(f"Horn slot spans Y in [{horn_slot_y_min:.2f}, {horn_slot_y_max:.2f}]")
print(f"  Bore center is Y=0, bore extends Y in [{-bore_r:.2f}, {bore_r:.2f}]")
print(f"  Pocket inner wall at Y={servo_y_inner:.2f}, pocket outer at Y={servo_y_outer:.2f}")
# Does it reach the pocket?
print(f"  Reaches pocket interior? {horn_slot_y_max >= servo_y_inner}  ({horn_slot_y_max:.2f} >= {servo_y_inner:.2f})")
print(f"  Reaches bore?            {horn_slot_y_min <= bore_r}  ({horn_slot_y_min:.2f} <= {bore_r:.2f})")


Servo pocket inner Y = 13.70, bore radius = 12.70, residual wall = 1.00 mm
Horn slot spans Y in [-6.35, 20.05]
  Bore center is Y=0, bore extends Y in [-12.70, 12.70]
  Pocket inner wall at Y=13.70, pocket outer at Y=30.00
  Reaches pocket interior? True  (20.05 >= 13.70)
  Reaches bore?            True  (-6.35 <= 12.70)


In [3]:
# Horn slot Y range is [-6.35, 20.05]. Bore is [-12.7, 12.7]. Pocket inner wall at 13.70.
# So horn slot punches from Y=-6.35 to Y=20.05, which:
#  - enters the bore on one side only (it never reaches Y=-12.7, so it doesn't cut all the way through the bore)
#  - DOES reach into the pocket cavity (20.05 > 13.70)
# Good: it tunnels from pocket through to bore on +Y side.
# But check: the horn slot center is Y=6.85; it does NOT cross the bore axis cleanly,
# and the slot is rectangular - it cuts a chord-shaped piece out of the bore wall.
# The bore at Y in [-6.35, 12.7] (inside the slot) is now connected to pocket.
# That's fine kinematically for a servo horn.

# Now check: the servo "pivots the dispense point" claim.
# The dispense end is at Z=0. The servo is at servo_pocket_z_center = 25 mm.
# So the horn pushes the auger tube *sideways* at Z=25 mm from the dispense end.
# But the auger tube is encased in the chassis monolith! It is NOT a separate
# pivoting part. There is no hinge, no flexure, no separate dispense nozzle
# coupled to the servo horn. The "pivot the dispense point" is geometrically
# impossible without a separate hinged dispense nozzle.

# Let's verify: is there any mention of a flexure, hinge, or separate spout part?
# Look at the part: chassis is a single solid body. Auger is a separate insertable
# helix. The dispense end is the bottom of the chassis bore. The servo can push
# on... the auger flute? But the auger is rotating!

# This is a major architectural confusion. Note it.
print("ARCHITECTURE CHECK:")
print("  Chassis is a monolithic block with a bore through it.")
print("  Auger is a rotating part inside the bore.")
print("  Servo horn enters bore via horn slot at z=25, but there is no separate")
print("  hinged 'dispense spout' part for the servo to actuate.")
print("  → 'pivot the dispense point so spout stays stationary' has no physical mechanism.")


ARCHITECTURE CHECK:
  Chassis is a monolithic block with a bore through it.
  Auger is a rotating part inside the bore.
  Servo horn enters bore via horn slot at z=25, but there is no separate
  hinged 'dispense spout' part for the servo to actuate.
  → 'pivot the dispense point so spout stays stationary' has no physical mechanism.


In [4]:
# ---- Auger cantilever / whip check ----
# Printed PLA shaft, 250 mm long, ~6 mm OD (shaft_r=3), with helical fin.
# Cantilevered from one end (top coupler), no bottom bearing.

# Estimate first bending natural frequency for a cantilevered round beam:
#   f1 = (1.875^2 / (2π)) * sqrt(E*I / (ρ*A*L^4))
# Use the shaft only (conservative; the fin adds some stiffness and a lot of mass).
import math
L = 0.250          # m
d = 0.006          # m (shaft OD)
r = d/2
A = math.pi*r**2
I = math.pi*r**4/4
E_PLA = 3.5e9       # Pa (PLA typical 2.5–4 GPa)
rho_PLA = 1240      # kg/m^3
beta1 = 1.875
f1 = (beta1**2 / (2*math.pi)) * math.sqrt(E_PLA*I / (rho_PLA*A*L**4))
print(f"Cantilever 1st natural freq (shaft only, PLA): f1 ≈ {f1:.1f} Hz")

# Static tip deflection under 0.1 N lateral load (e.g., bore drag, eccentricity):
F = 0.1   # N
delta = F*L**3 / (3*E_PLA*I)
print(f"Static tip deflection under {F} N lateral: δ ≈ {delta*1000:.2f} mm")

# Stepper speed regime — NEMA-11 microstepping at e.g. 1 rev/s = 60 rpm = 1 Hz.
# Fast dosing up to 5 rev/s = 5 Hz. Both well below f1 ~ tens of Hz, so fundamental
# bending mode is unlikely to be directly excited by rotation rate.
# BUT: ERM disc has a typical operating frequency of 150-250 Hz (9000-15000 rpm).
# That is FAR above the auger bending mode — won't directly couple.
# However, structural resonances of the chassis tube (much larger, more PLA mass)
# will sit in the tens-of-Hz to low-hundreds range and the ERM can absolutely hit
# those.

# Critical whirl: for a cantilever, whirl speed ≈ first bending freq (in Hz).
# So whirl speed in rpm ≈ 60 * f1
print(f"Whirl onset speed ≈ {60*f1:.0f} rpm (compare to expected 60-300 rpm)")


Cantilever 1st natural freq (shaft only, PLA): f1 ≈ 22.6 Hz
Static tip deflection under 0.1 N lateral: δ ≈ 2.34 mm
Whirl onset speed ≈ 1354 rpm (compare to expected 60-300 rpm)


In [5]:
# So whirl onset at ~1350 rpm — auger likely OK for typical dosing rotation rates (60-300 rpm).
# BUT: under any radial preload (powder packing, off-axis grub-screw tightening, the
# fact that the helix is a separate Compound that may not be coaxial with the shaft —
# remember the auger script ships the fin as a Compound, NOT fused, so geometry may
# rely on print-bed alignment of two separately-printed pieces? Let's re-read.)

# Actually rereading: the .Compound is a CAD trick to keep the fin visible. When
# slicing this, the slicer will *probably* still print both bodies because slicers
# walk all solids in the input mesh. But the fin is a thin (2 mm) helical surface
# that intersects the shaft — if the slicer treats them as separate bodies, you get
# z-fighting at the intersection, possibly air gaps, possibly retraction artifacts.
# The user explicitly notes the FALLBACK is hand-coded after CADSmith failed to
# manifold-fuse the helix.

# Static deflection of 2.3 mm under just 0.1 N tells us the auger is extremely
# compliant laterally. Powder column drag alone could push the tip into the bore
# wall (clearance = (auger_bore_d - auger_tube_outer_d)/2 = 0.2 mm).
clearance = (auger_bore_d - auger_tube_outer_d)/2
print(f"Radial clearance auger-to-bore = {clearance:.2f} mm")
print(f"Force to deflect tip by clearance: F = {3*E_PLA*I*clearance/1000 / L**3 * 1000:.3f} N")
# i.e., a tiny lateral force makes the tip rub.

# But hold on — the AUGER OD in the auger script is the *outer* tube_body
# (cylinder of radius outer_r = 12.5). So the auger is the full 25 mm OD tube,
# not a 6 mm shaft! Let me re-read.


Radial clearance auger-to-bore = 0.20 mm
Force to deflect tip by clearance: F = 8.550 N


In [6]:
# WAIT — re-reading the auger script carefully:
#   "result = cq.Workplane('XY').circle(outer_r).extrude(auger_tube_length)"
# outer_r = 12.5 mm. So the auger is a SOLID 25mm OD cylinder, 250mm tall.
# Then a bore (inner_r = 10.5 mm) is cut through ~from z=funnel_h to z=tube_length-top_cap_h.
# That makes the auger a 25 mm OD tube with 2 mm walls. So it's a *tube*, with
# an internal 6mm shaft, and a helical fin sweeping out to inner_r + fin_sink.
#
# But this is then DROPPED INTO the chassis bore (which is also 25.4 mm)!
# Outer surface of auger (Ø25) rides inside chassis bore (Ø25.4) with 0.2 mm clearance.
# So the *auger itself* is what carries powder — outside the auger? Or inside?
#
# Looking at the helix: helix_R = (shaft_r + (inner_r + fin_sink))/2 = (3 + 10.9)/2 = 6.95
# helix max radius = inner_r + fin_sink = 10.9 ≈ wall ID 10.5
# So the helix is INSIDE the auger's own internal bore. Powder flows DOWN the
# INSIDE of the auger (between the central 6mm shaft and the 21mm internal bore wall,
# with a helical fin moving it).
#
# Wait, but then the auger BODY is also the bore wall — and it rotates! If the body
# rotates, so does the "bore wall" the helix sweeps powder against. That means
# the entire auger-and-tube assembly rotates as one piece, with helix and outer
# wall co-rotating. Powder rides the helix down? But the powder can't grip both
# the wall (rotating) AND the helix (rotating with same wall) → no relative motion
# → no metering!
#
# This is a fundamental Archimedes-screw error. An auger needs a STATIONARY housing
# tube and a ROTATING helix inside it. The relative motion between rotating helix
# and stationary wall is what advances the powder. If the helix and the wall rotate
# together (no relative motion), powder either doesn't move or spins as a rigid body.

print("AUGER FUNDAMENTAL ERROR:")
print(f"  Auger is a Ø{auger_tube_outer_d:.1f} mm tube (2 mm wall) that rotates inside the Ø{auger_bore_d:.1f} mm chassis bore.")
print(f"  Helical fin is INSIDE the auger's own internal bore (Ø21).")
print(f"  → fin and wall co-rotate → no relative motion to advance powder.")
print(f"  → architecture conflates 'auger tube' (housing) with 'auger screw' (rotor).")


AUGER FUNDAMENTAL ERROR:
  Auger is a Ø25.0 mm tube (2 mm wall) that rotates inside the Ø25.4 mm chassis bore.
  Helical fin is INSIDE the auger's own internal bore (Ø21).
  → fin and wall co-rotate → no relative motion to advance powder.
  → architecture conflates 'auger tube' (housing) with 'auger screw' (rotor).


In [7]:
# Let me also check ERM coupling and the pause embedding sanity.

# ERM pause is at z=125 mm. ERM pocket is at z=125 (mid tube).
erm_pocket_z = 125.0  # auger_tube_length * 0.5
erm_pocket_d = 10.3
erm_pocket_depth = 2.9   # erm_disc_thk + 0.2
# Pocket extends from x = chassis_x/2 inward by 2.9 mm — i.e., from x=35 to x=32.1
# The ERM coin is a Ø10 disc 2.7 mm thick. It is potted INTO this pocket. After
# resume, plastic is laid down ON TOP OF the coin (the next layer at z=125.2 mm is
# directly above the coin top surface). The coin is then fully encapsulated in
# the chassis wall, 32 mm from the bore axis (well, from x=32.1 inward to bore at x=12.7,
# = 19.4 mm of solid plastic between the coin face and the bore wall).
plastic_between_coin_and_bore = 32.1 - 12.7
print(f"Plastic mass between ERM coin face and bore wall: {plastic_between_coin_and_bore:.1f} mm")

# Pause Z=125 means everything from z=0 to z=125 has already been printed when the
# coin goes in. The coin sits in a 2.9 mm deep open pocket. Then resume prints
# layers AT z=125.2+, which means... it bridges OVER the open pocket? No - the
# pocket is in the XY direction (it's a radial cut). So at z=125, the cross-section
# at z just below 125 already shows the pocket as a hole in the wall going outward.
# At z=125+erm_pocket_thk/2, the cross-section starts to close back over the coin.

# Actually the pocket is cut perpendicular to the X axis as a cylinder of axis X,
# radius 5.15, depth 2.9 (in -X direction from x=35). So the pocket as a function
# of Z opens between z=125-5.15 and z=125+5.15 (the cylindrical cross-section is
# a circle in the YZ plane). The pause must be timed for when the pocket TOP is
# being printed so the coin can be dropped in.

# Z of TOP of pocket = 125 + erm_pocket_d/2 = 125 + 5.15 = 130.15 mm. So if you
# pause at z=125 (which is the MIDDLE of the pocket), the pocket is half-printed
# (the top half is already a U-shaped opening). Hmm — let me think again.
# The pocket is a horizontal hole in the +X face. Looking at a YZ cross-section
# at any X in [32.1, 35]: there's a circular hole of radius 5.15 centered at z=125.
# As you print bottom-up, you start cutting into the pocket at z = 125-5.15 = 119.85
# and finish at z=125+5.15 = 130.15.
# So at z=125 the pocket is a horizontal slot — the bottom half is open, the top
# half is yet to be printed (bridged over).
# To insert the coin, you pause when the BOTTOM of the pocket has been printed
# (z=120) but you'd want the coin to drop into a *complete bottom half* of the
# pocket. The recommended pause Z=125 actually has the printer in the middle of
# the bridge — the bottom half of the pocket is a U-shaped channel. The coin
# (2.7 mm thick) needs a flat surface to land on; the bottom of the pocket is
# at y=0 (cylinder center), but the bottom of the U-channel at z=125 is the bottom
# arc of the cylinder.

# Bottom line: ERM pocket should be a SQUARE or RECTANGULAR pocket (slot), not
# a circular cylinder, so the coin lands flat. As currently designed, the coin
# would have to be glued/centered on a curved bottom.

print("\nERM POCKET ISSUE:")
print("  Pocket is a Ø10.3 × 2.9 mm circular cylinder (axis along +X).")
print("  At pause Z=125 the lower half of the pocket forms a U-shaped channel.")
print("  A 10mm disc dropped in does NOT sit on a flat surface — it rests at the")
print("  bottom of a cylindrical depression. Encapsulation by resume requires")
print("  bridging over the top half of the pocket + the coin (~5 mm bridge), which")
print("  on most FDM printers will sag without support.")


Plastic mass between ERM coin face and bore wall: 19.4 mm

ERM POCKET ISSUE:
  Pocket is a Ø10.3 × 2.9 mm circular cylinder (axis along +X).
  At pause Z=125 the lower half of the pocket forms a U-shaped channel.
  A 10mm disc dropped in does NOT sit on a flat surface — it rests at the
  bottom of a cylindrical depression. Encapsulation by resume requires
  bridging over the top half of the pocket + the coin (~5 mm bridge), which
  on most FDM printers will sag without support.


In [8]:
# ERM coupling through PLA. The user asks: "will the surrounding plastic damp it
# to nothing?" Quick first-pass:
#
# ERM with eccentric mass m at radius r0 spinning at ω generates radial force
# F = m * r0 * ω^2.
# Adafruit 1201: nominal 0.8-1.0 G at 12000 rpm for the bare disc (free body ~1 g).
# When embedded into a heavy chassis, the chassis becomes the inertial reference;
# the relevant transmissibility depends on |F_excitation / m_chassis * ω^2| at the
# point you care about (the bore).
#
# Chassis mass: 70×60×300 mm box minus ~Ø25 bore minus e-bay cavity.
# Quick estimate:
import math
V_chassis = 70 * 60 * 300                   # mm^3 outer envelope
V_bore = math.pi * (25.4/2)**2 * 250        # bore void
V_ebay_cav = 65 * 32.5 * 75                  # e-bay void
V_solid = V_chassis - V_bore - V_ebay_cav   # mm^3
m_chassis_g = V_solid * 1.24e-3              # PLA density 1.24 g/cm^3 = 1.24e-3 g/mm^3
print(f"Chassis solid volume ≈ {V_solid/1000:.0f} cm³, mass ≈ {m_chassis_g:.0f} g (assuming 100% infill)")
# At ~30% infill typical FDM:
print(f"Chassis at 30% infill ≈ {m_chassis_g*0.4:.0f} g (walls + infill weighted)")

# ERM excitation: F ≈ 1 G of its own mass. Bare ERM = 0.9 g, so F ≈ 9 mN.
# Acceleration imparted to a 600 g chassis: a = F/M = 0.009/0.6 = 0.015 m/s^2 ≈ 1.5 mG.
# That's 600x less than the bare ERM in free air.
# This is precisely why ERMs are normally surface-mounted on a thin compliant layer,
# not buried in 19 mm of PLA.
print("\nERM COUPLING ANALYSIS:")
print(f"  Bare ERM peak force ~9 mN at 12k rpm (≈1 G of its own ~0.9 g mass).")
print(f"  Coupled to ~600 g chassis monolith → bore-wall acceleration ≈ 1.5 mG.")
print(f"  Typical powder fluidization threshold ≈ 0.5-2 G of local vibration.")
print(f"  → Encapsulated ERM at this distance is ≥100× under-spec'd for de-bridging powder.")


Chassis solid volume ≈ 975 cm³, mass ≈ 1209 g (assuming 100% infill)
Chassis at 30% infill ≈ 484 g (walls + infill weighted)

ERM COUPLING ANALYSIS:
  Bare ERM peak force ~9 mN at 12k rpm (≈1 G of its own ~0.9 g mass).
  Coupled to ~600 g chassis monolith → bore-wall acceleration ≈ 1.5 mG.
  Typical powder fluidization threshold ≈ 0.5-2 G of local vibration.
  → Encapsulated ERM at this distance is ≥100× under-spec'd for de-bridging powder.


In [9]:
# Solenoid heat / encapsulation. Adafruit #412 5V solenoid draws ~1.1 A on pull-in,
# but the datasheet rates it for SHORT DUTY (max ~30% duty cycle at 5V).
# Continuous-on dissipation: P = V*I ≈ 5 * 1.1 = 5.5 W.
# Encapsulated in PLA (which softens at 55–60 °C glass transition, deforms at 70 °C),
# steady-state surface temp rises ~T = P/(h*A). With h ≈ 5 W/m²K (still air) and
# coil surface area ≈ 9.6×19×4 sides ≈ 1500 mm² = 1.5e-3 m²:
T_rise = 5.5 / (5 * 1.5e-3)
print(f"Solenoid steady-state ΔT in PLA encapsulation (continuous on): ≈ {T_rise:.0f} K")
# That would melt PLA. Even at 10% duty (typical for a tap-style action):
# P_avg = 0.55 W → ΔT ≈ 70 K, marginal.
# At 1% duty (brief tap), P_avg = 0.055 W, ΔT ≈ 7 K — fine.
# Conclusion: solenoid is OK ONLY if duty cycle is kept very low. Stuck-on
# failure mode = chassis warp / melt. No fuse or thermal cutout in design.

# Tic500 + Bonnet overlap (electrical architecture):
# Adafruit Stepper Bonnet uses TB6612 dual H-bridges, max ~1.2 A/coil at 12 V.
# Pololu Tic 500 uses TI DRV8434, up to 1.5 A/coil, with USB/I2C/serial control.
# Both can drive a NEMA-11 (typ. 0.67 A/phase for 11HS18-0674S).
# The Bonnet's I2C addresses are PCA9685-derived (0x60-0x7F). Tic 500 default I2C
# address is 0x0E. No conflict by default. But the Bonnet OCCUPIES THE PI GPIO
# HEADER fully (it's a HAT), so the Tic 500 must connect via USB (not I2C),
# OR the Bonnet header must be jumpered to break out I2C separately.
# This needs clarifying.

# Also: ERM and solenoid are DC loads; the Bonnet's TB6612 H-bridges CAN drive
# them, but the second motor channel must then be dedicated to one of them.
# Bonnet has 2 stepper channels (= 4 DC channels). Allocations:
#   stepper (4 DC channels worth) — but we said Tic500 drives stepper.
#   So Bonnet has 4 DC channels free → ERM, solenoid, and 2 spare. OK.
# But servo is PWM @ 50 Hz, which the Bonnet ALSO supports (PCA9685 on board).
# Good — Bonnet handles ERM (PWM/DC), solenoid (DC), servo (50Hz PWM). Tic 500
# handles stepper. Architecture is workable; not "double-counting" — the DRV8825
# is the redundancy.

print("\nDRIVER ARCHITECTURE:")
print("  Bonnet: ERM (PWM), solenoid (DC), servo (50Hz). Tic500: stepper (USB).")
print("  DRV8825 is genuinely redundant (fallback). Not a bus conflict, but")
print("  the Tic500 cannot use I2C if the Bonnet occupies the GPIO header.")
print("  Default connection is USB; needs explicit USB cable routing.")


Solenoid steady-state ΔT in PLA encapsulation (continuous on): ≈ 733 K

DRIVER ARCHITECTURE:
  Bonnet: ERM (PWM), solenoid (DC), servo (50Hz). Tic500: stepper (USB).
  DRV8825 is genuinely redundant (fallback). Not a bus conflict, but
  the Tic500 cannot use I2C if the Bonnet occupies the GPIO header.
  Default connection is USB; needs explicit USB cable routing.


In [10]:
# Stepper plate hole pattern check:
# Adafruit lists 11HS18-0674S as a NEMA-11 with 23 × 23 mm M2.5 bolt pattern, Ø22 pilot.
# The code uses stepper_bolt_pattern = 23.0, stepper_pilot_d = 22.0 — matches. ✓

# Coupler chamber:
# ST-FC01 5–5 coupler is typically 19 × 25 mm. Chamber is Ø19 × 26 mm. Snug, OK.
# Grub-screw access ports: Ø3.5 mm at z_low = (300 - 26 + 6) = 280, z_high = 300 - 6 = 294.
# Set screws on ST-FC01 are at ~6 mm from each end → screws at z=286 (auger side, 6mm above coupler bottom 280)
# and z=294 (stepper side, 6mm below coupler top 300). Code places ports at z=280 and z=294.
# The auger-side port is 6 mm too low (z=280 is the bottom of the chamber, not the screw).
# Should be z = coupler_z_bottom + 6 = 274 + 6 = 280 — wait, coupler_z_bottom = 300 - 26 = 274.
# grub_z_low = 274 + 6 = 280. So port is at z=280, screw is at z=274+6=280. ✓ Match.
# Wait — re-check:
chassis_z = 300
coupler_chamber_h = 26
coupler_z_bottom = chassis_z - coupler_chamber_h   # 274
coupler_z_top = chassis_z                            # 300
grub_z_low = chassis_z - coupler_chamber_h + 6.0    # 280
grub_z_high = chassis_z - 6.0                        # 294
print(f"Coupler chamber: z ∈ [{coupler_z_bottom}, {coupler_z_top}]")
print(f"Grub ports at z = {grub_z_low}, {grub_z_high}")
print(f"ST-FC01 setscrews ~6mm from each end → at z={coupler_z_bottom+6}, z={coupler_z_top-6}")
print(f"Alignment: {'OK' if (grub_z_low == coupler_z_bottom+6 and grub_z_high == coupler_z_top-6) else 'MISALIGNED'}")
# OK ✓

# Now — one more potentially fatal issue: the stepper plate covers the top of the
# auger bore. The auger has to be dropped into the chassis from the TOP. But the
# stepper_plate is unioned ON TOP at z=300 with a Ø22 pilot hole. The auger OD is
# Ø25. So you can't insert the auger from the top after printing!
plate_pilot = 22
auger_od = 25
print(f"\nAuger insertion check: plate pilot Ø{plate_pilot}, auger OD Ø{auger_od}")
print(f"  → auger DOES NOT FIT through plate pilot. Must be inserted from below (Z=0).")
# But the chassis BORE only goes from z=0 to z=auger_tube_length=250. Above z=250
# the chassis is solid except for the coupler chamber (Ø19, z=274-300).
# Between z=250 and z=274 there is SOLID PLASTIC (a ledge between bore Ø25.4
# and coupler chamber Ø19). So the auger top (which has an M3 stub) must engage
# the coupler from below. Fine.
# But to install from below, the dispense end of the chassis must be open at z=0.
# It is — the bore goes all the way through (auger_bore extruded from z=0).
# So the auger goes in from the dispense (bottom) end. OK, but: the auger has a
# top cap with loading slots — the cap is Ø25 (the same as outer_r=12.5). After
# insertion from below, the top cap sits at z=250, and the M3 stub above it
# engages the coupler. But wait — the boss is at z=auger_tube_length - top_cap_h - boss_h
# = 250 - 6 - 6 = 238. So the M3 stub is at z=238? No, it's the boss BELOW the cap.
# The auger top is at z=auger_tube_length = 250. So the top of the auger is at z=250.
# The coupler chamber starts at z=274. There's a 24 mm gap.
# This means the auger's M3 stub has to reach UP from z=250 to engage a coupler
# that's at z=274-300. The auger is described as having an "M3 spindle" but the
# script doesn't extrude a spindle above the top cap.

print(f"\nAuger top at z=250, coupler chamber at z=274-300. GAP of 24 mm.")
print(f"Auger script does NOT extrude an M3 spindle above the top cap (z=250).")
print(f"  → Even if you insert the auger, there is nothing to grip with the ST-FC01.")


Coupler chamber: z ∈ [274, 300]
Grub ports at z = 280.0, 294.0
ST-FC01 setscrews ~6mm from each end → at z=280, z=294
Alignment: OK

Auger insertion check: plate pilot Ø22, auger OD Ø25
  → auger DOES NOT FIT through plate pilot. Must be inserted from below (Z=0).

Auger top at z=250, coupler chamber at z=274-300. GAP of 24 mm.
Auger script does NOT extrude an M3 spindle above the top cap (z=250).
  → Even if you insert the auger, there is nothing to grip with the ST-FC01.


In [11]:
# Solenoid pocket footprint check:
solenoid_pocket_x = 9.8
solenoid_pocket_y = 19.2
solenoid_pocket_z_dim = 22.2
solenoid_pocket_z_center = 30.0
# Pocket extends from x = -35 to x = -35 + 9.8 = -25.2 (inward from -X face).
# Slot extends from x = -25.2 to bore wall (x ≈ -12.7).
# Distance from slot inner end to bore wall: -25.2 - (-12.7) = 12.5 mm of plunger travel
slot_travel = 25.2 - 12.7
print(f"Solenoid plunger must travel ~{slot_travel:.1f} mm from pocket inner wall to reach bore.")
# Adafruit #412 JF-0530B plunger stroke is ~4 mm.
print(f"  JF-0530B has only ~4 mm of stroke. → plunger CANNOT reach bore by ~8 mm.")

# Solenoid pocket Z extent:
sol_z_min = solenoid_pocket_z_center - solenoid_pocket_z_dim/2
sol_z_max = solenoid_pocket_z_center + solenoid_pocket_z_dim/2
print(f"Solenoid pocket Z range: [{sol_z_min:.1f}, {sol_z_max:.1f}]")
# Pause is at z=41.1 — within the pocket. Note that the pocket EXTENDS BELOW the
# pause Z. From z=18.9 to z=41.1 (above pause), pocket is open.
# That means at z=41.1 pause, the printer has printed from z=0 to z=41.1, including
# the BOTTOM HALF of the solenoid pocket (z=18.9 to 41.1). The pocket has been
# carved out continuously throughout that range. So at pause time, the bottom 22 mm
# of the pocket is fully open — solenoid drops in flush.
# ABOVE the pause, layers from z=41.1 to z=41.3 will be the encapsulation layer,
# but wait — the pocket extends up to z=41.1 only. So the pause IS at the top of
# the pocket. ✓ Good timing for solenoid.

# But ERM pocket centered at z=125 with depth 5.15 in Z means pocket goes from
# z=119.85 to z=130.15. Pause at z=125 is the MIDDLE — not the top — which means
# 5.15 mm of pocket top is yet to be printed (bridging).
print(f"\nERM pocket Z range: [119.85, 130.15], pause at z=125")
print(f"  → 5.15 mm of unsupported bridge over the coin after resume. Not ideal.")
print(f"  Better: pause at z = 130.15 (top of pocket) so coin drops in and is immediately encapsulated by next layer.")


Solenoid plunger must travel ~12.5 mm from pocket inner wall to reach bore.
  JF-0530B has only ~4 mm of stroke. → plunger CANNOT reach bore by ~8 mm.
Solenoid pocket Z range: [18.9, 41.1]

ERM pocket Z range: [119.85, 130.15], pause at z=125
  → 5.15 mm of unsupported bridge over the coin after resume. Not ideal.
  Better: pause at z = 130.15 (top of pocket) so coin drops in and is immediately encapsulated by next layer.


In [12]:
# Wait — let me reconsider. The JF-0530B is described as 9.6 x 19 x 22 mm, plunger
# being roughly the "long" 22mm axis. The plunger length ITSELF protrudes from the
# coil body. Adafruit page: "5mm of throw" — and the plunger sticks out ~10 mm
# at rest, retracts ~5 mm. So total reach from the pocket inner face = plunger
# protrusion (~10mm at rest) + throw (~5 mm) = ~15 mm of forward reach.
# The slot from pocket-inner to bore is 12.5 mm. So plunger reaches.
# Hmm — depends on plunger protrusion which varies. Let me not over-call this one.
# But still flag it as "verify plunger reach > 12.5 mm before printing."

# More importantly: the solenoid mounting orientation:
# Pocket is 9.8 (X) × 19.2 (Y) × 22.2 (Z). Solenoid body is 9.6 × 19 × 22 mm.
# In the JF-0530B, the plunger axis is along the 22 mm dimension.
# In the pocket, the 22.2 mm axis is along Z (vertical). So the plunger would
# point in +Z or -Z, NOT toward the bore (-X to +X). The slot is cut along X
# (slot_w = 2.5 along Y, slot_h = 6 along Z, length = 25.2 mm along X).
# So the plunger orientation is fundamentally MISALIGNED with the slot.

print("SOLENOID ORIENTATION ERROR:")
print("  Pocket dimensions (X,Y,Z) = (9.8, 19.2, 22.2)")
print("  Solenoid body (W,H,L) = (9.6, 19, 22); plunger axis along the 22 mm dimension")
print("  → Pocket forces plunger axis along Z (vertical), but slot to bore is along X.")
print("  Must rotate pocket so 22.2 axis lies along X (plunger toward bore).")
print("  Correct pocket dims: (22.2, 19.2, 9.8) along (X, Y, Z).")


SOLENOID ORIENTATION ERROR:
  Pocket dimensions (X,Y,Z) = (9.8, 19.2, 22.2)
  Solenoid body (W,H,L) = (9.6, 19, 22); plunger axis along the 22 mm dimension
  → Pocket forces plunger axis along Z (vertical), but slot to bore is along X.
  Must rotate pocket so 22.2 axis lies along X (plunger toward bore).
  Correct pocket dims: (22.2, 19.2, 9.8) along (X, Y, Z).


In [13]:
# Pi cable egress and e-bay opening sanity:
# The bay lid opening is cut on the -Y face: 
ebay_lid_opening_w = 65 - 2
ebay_lid_opening_h = 75 - 2
# wait — ebay_cav_x = 70 - 2*2.5 = 65, ebay_cav_z = 80 - 2*2.5 = 75
# Opening is the FULL cavity minus 2mm border — essentially the entire -Y wall is removed.
# So the e-bay wall thickness 2.5mm is ENTIRELY cut away on -Y? Then where do the
# lid screws thread INTO? The heat-set inserts are cut at z offset HEATSET_M3_DEP=5.5
# from ebay_y_outer_face — meaning ~5.5 mm into the -Y wall. But the -Y wall was
# just removed by the lid opening cut. Let's check:
ebay_outer_y = 35.0
ebay_wall_thk = 2.5
# wall_thk = 2.5 mm — the -Y wall is 2.5 mm thick. The opening is cut 2.5+0.1 deep.
# So yes, the ENTIRE -Y wall is removed by the lid opening cut. Heat-set inserts
# then attempt to drill 5.5 mm into... nothing? They'd be drilling into the
# *side* walls' end faces. Let me re-read the lid_inserts code more carefully.

# lid_inserts is on plane XZ at offset = ebay_y_outer_face + HEATSET_M3_DEP
# = -60 - 35 + 5.5 = -89.5. That's the y position of the back of the insert hole.
# pushPoints uses (x, z) coordinates on the XZ plane. The hole circles are at
# corners of the e-bay outer face. extruded by -5.5 - 0.1 in -Y direction.
# So holes are at y in [-89.5 - 5.5 -0.1 = -95.1, -89.5] → y in [-95.1, -89.5].
# E-bay outer face is at y = ebay_y_outer_face = -95. E-bay's -Y face is at -95.
# So holes from y=-95.1 to y=-89.5 = from outer face inward by 5.5 mm.
# BUT the lid opening cuts away the -Y wall (y=-95 to y=-92.5 = -95 + 2.5).
# So holes are drilled through the (already missing) -Y wall and into y up to -89.5,
# which is INSIDE THE CAVITY (cavity starts at -92.5).
# Result: lid_inserts are floating in space inside the cavity, not in any material.

print("E-BAY LID INSERT ISSUE:")
print("  -Y wall of e-bay is 2.5 mm thick (y ∈ [-95, -92.5]).")
print("  Lid opening cut removes 2.5+0.1 mm of -Y wall → entire -Y wall gone.")
print("  Lid insert holes are drilled at y ∈ [-95.1, -89.5] — into now-empty cavity.")
print("  → Lid cannot be screwed on; the screw holes don't exist in any material.")


E-BAY LID INSERT ISSUE:
  -Y wall of e-bay is 2.5 mm thick (y ∈ [-95, -92.5]).
  Lid opening cut removes 2.5+0.1 mm of -Y wall → entire -Y wall gone.
  Lid insert holes are drilled at y ∈ [-95.1, -89.5] — into now-empty cavity.
  → Lid cannot be screwed on; the screw holes don't exist in any material.


In [14]:
# Hmm — actually wait. The lid opening rect is centered at (0, ebay_cav_z_min + ebay_cav_z/2)
# with width ebay_lid_opening_w = 63 mm (= 65 - 2). The e-bay outer x = 70 mm, cavity
# x = 65 mm. So the lid opening (63 wide) does NOT extend to the full e-bay outer width.
# That leaves 3.5 mm of -Y wall on each side... wait but the -Y wall is only 2.5 mm
# thick (in Y), not in X. The opening is a rectangle cut in XZ plane, extruded in Y.
# The wall is in Y. The opening removes a rectangle 63 wide x 73 tall from the -Y wall.
# Outside this rectangle, the -Y wall still exists. But the lid_insert holes are at
# corners of the e-bay OUTER face — let me check their (x, z) positions:
ebay_outer_x = 70
ebay_z_min = 50
ebay_z_max = 130
ebay_lid_inset = 4.0
lid_hole_pts = [
    ( ebay_outer_x/2 - ebay_lid_inset,  ebay_z_min + ebay_lid_inset),   # (31, 54)
    (-ebay_outer_x/2 + ebay_lid_inset,  ebay_z_min + ebay_lid_inset),   # (-31, 54)
    ( ebay_outer_x/2 - ebay_lid_inset,  ebay_z_max - ebay_lid_inset),   # (31, 126)
    (-ebay_outer_x/2 + ebay_lid_inset,  ebay_z_max - ebay_lid_inset),   # (-31, 126)
]
# Opening rectangle: width 63 (x ∈ [-31.5, 31.5]), height 73 (z ∈ [54, 127] approx)
# Hole x positions: ±31. These are INSIDE the 63-wide opening (|31|<31.5).
# Hole z positions: 54 and 126. Opening z spans from ebay_cav_z_min + ebay_cav_z/2 - 73/2
# = (50+2.5) + 75/2 - 36.5 = 52.5 + 37.5 - 36.5 = 53.5 to 53.5+73 = 126.5.
# So holes at z=54 and z=126 are INSIDE the opening (53.5 < 54, 126 < 126.5).
# Confirmed: all 4 lid screw holes are inside the lid opening cutout → screws don't engage material.

print(f"Lid hole positions (x, z): {lid_hole_pts}")
print(f"Lid opening: x ∈ [-31.5, 31.5], z ∈ [53.5, 126.5]")
print(f"All 4 holes are INSIDE the opening → no thread engagement possible.")


Lid hole positions (x, z): [(31.0, 54.0), (-31.0, 54.0), (31.0, 126.0), (-31.0, 126.0)]
Lid opening: x ∈ [-31.5, 31.5], z ∈ [53.5, 126.5]
All 4 holes are INSIDE the opening → no thread engagement possible.


In [15]:
# Check the Pi cable slot interaction with the bay floor:
ebay_z_min = 50
ebay_outer_y = 35
ebay_wall_thk = 2.5
ebay_y_outer_face = -30 - 35  # -65? Wait: ebay_y_outer_face = -chassis_y/2 - ebay_outer_y = -30 - 35 = -65
print(f"ebay_y_outer_face = {-30-35}")
# pi_cable_slot at ebay_z_min - 0.1 = 49.9, extends 2.7 in Z.
# center y = ebay_y_outer_face + ebay_wall_thk + pi_cable_slot_d/2
pi_cable_slot_d = ebay_outer_y - ebay_wall_thk   # 32.5
print(f"slot center y = {-65 + 2.5 + 32.5/2}")  
# Slot width 50 (X), depth 32.5 (Y), height 2.7 (Z) at z=50
# This slot cuts the FLOOR of the e-bay. OK in principle. But the Pi is mounted on
# the +Y inner wall and connectors face -Z; the connector edge of the Pi is at the
# BOTTOM Z position of the Pi board. Pi pattern z_center = ebay_z_min + ebay_outer_z/2 = 90.
# Pi spans 65×30 with 58×23 mounting pattern. Pi z-extent: 90 ± 15 = [75, 105].
# Pi connector edge at z=75. Cable slot at z=50. Cables must traverse ~25 mm of free air
# inside the bay before exiting through the slot. Doable but messy.

# Critical check: does the e-bay cavity sidewall intersect with the auger bore?
# E-bay cavity X extent: ±32.5. Chassis main body X extent: ±35.
# Auger bore is at x=0, radius 12.7. Bore doesn't reach the e-bay (well below x=32.5).
# BUT the e-bay cavity extends in +Y from y = -95 + 2.5 = -92.5 up to
# y = ebay_cav_y_center + ebay_cav_y/2.
ebay_cav_y = ebay_outer_y - ebay_wall_thk   # 32.5
ebay_cav_y_center = -95 + 2.5 + 32.5/2     # -76.25
ebay_cav_y_max = ebay_cav_y_center + ebay_cav_y/2   # -60
print(f"E-bay cavity y range: [{-95+2.5}, {ebay_cav_y_max}]")
# Cavity stops at y=-60. Chassis -Y face is at y=-30. So there's still ~30 mm of
# chassis material between the cavity and the bore. OK.

# But the GROMMET hole: grommet is at z=grommet_z = ebay_z_max - 8 = 122,
# on the chassis -Y face (y=-30), extruded in -Y direction by ebay_wall_thk + 1 = 3.5 mm.
# So grommet hole spans y in [-33.5, -30]. The e-bay cavity starts at y=-92.5.
# Distance from grommet outer end (y=-33.5) to e-bay cavity (y=-92.5) = 59 mm.
# The grommet does NOT actually connect into the e-bay — it just makes a dimple
# 3.5 mm deep into the chassis -Y face.

print("\nGROMMET ISSUE:")
print("  Grommet extruded 3.5 mm into -Y face of chassis (y ∈ [-33.5, -30]).")
print("  E-bay cavity starts at y = -92.5.")
print("  → 59 mm gap of solid plastic between grommet end and bay cavity.")
print("  → Stepper leads cannot reach the bay through the grommet.")


ebay_y_outer_face = -65
slot center y = -46.25
E-bay cavity y range: [-92.5, -60.0]

GROMMET ISSUE:
  Grommet extruded 3.5 mm into -Y face of chassis (y ∈ [-33.5, -30]).
  E-bay cavity starts at y = -92.5.
  → 59 mm gap of solid plastic between grommet end and bay cavity.
  → Stepper leads cannot reach the bay through the grommet.


In [16]:
# Wait — I need to recheck my e-bay placement. Let me re-read the code:
# ebay_y_outer_face = -chassis_y / 2.0 - ebay_outer_y  # = -30 - 35 = -65
# ebay_block centered at (0, ebay_y_outer_face + ebay_outer_y/2.0, ...)
#                       = (0, -65 + 17.5, ...) = (0, -47.5, ...)
# Box dimensions 70 × 35 × 80, so y extent = [-47.5 - 17.5, -47.5 + 17.5] = [-65, -30].

# So the e-bay's -Y face (outer) is at y=-65, and its +Y face is at y=-30, which
# is FLUSH with the chassis -Y face. Good.

# Cavity center: ebay_cav_y_center = ebay_y_outer_face + ebay_wall_thk + ebay_cav_y/2
#               = -65 + 2.5 + 16.25 = -46.25
# Cavity y extent: [-46.25 - 16.25, -46.25 + 16.25] = [-62.5, -30].

# Hmm — cavity y_max = -30 = chassis -Y face. That means the cavity is OPEN on
# the +Y side directly into the chassis interior? No — the chassis is solid PLA
# at y=-30 (the chassis -Y face); the cavity carves the e-bay interior up to its
# +Y wall, which IS the chassis -Y face. The cavity y_max = -30 means there's NO
# wall between cavity and chassis main body — it just butts against the main
# chassis -Y face. So the chassis material between cavity and bore is ~30 mm thick
# (from y=-30 to y=-12.7 bore wall), which is fine for separation.

# Grommet extruded -3.5 in -Y from y=-30: y ∈ [-33.5, -30]. That goes INTO the
# e-bay material (cavity is at y=-62.5 to -30; from y=-33.5 to -30 is still
# inside the cavity y range, but at z=122 which is also in the cavity z range).
# Hmm actually z=122 is in cavity z range [52.5, 127.5]. So at z=122, the cavity
# is open. The grommet at y∈[-33.5, -30] cuts an additional hole, but it's IN
# THE CAVITY (the cavity already removed material in this range).
# So actually the grommet is a non-op — the cavity already removes material there.

# But wait, the grommet's purpose is to make a hole in the chassis WALL (at y=-30
# in the +Y direction, i.e., into the main chassis). The current code extrudes
# in -Y direction (away from chassis into bay). To connect the bay to the +X wire
# slot, the grommet should go in +Y direction from the bay's +Y inner wall up to
# the +X wire slot inside the chassis main body. As written, it does nothing.

print("Confirmed: grommet hole is on the wrong side (extends -Y into the open bay")
print("cavity instead of +Y into the chassis main body where wires would arrive).")
print("And the +X wire slot inside the main chassis can't actually reach the bay")
print("on the -Y side without traversing the bore-side of the chassis.")

# Pi connector clearance:
# Pi standoffs raise Pi by 4 mm from the +Y inner wall (y = -30) into the cavity
# (in -Y direction). Pi sits at y ≈ -34. Pi thickness ~1.5 mm + components ~4 mm
# (HDMI, USB connectors ~6 mm), so Pi+stack envelope: y ∈ [-34 - 8 - bonnet stack 25, -34].
# Bonnet is ~25 mm tall on GPIO. GPIO is on top of Pi. So Pi+Bonnet stack:
# y ∈ [-34 - 25, -34] = [-59, -34].
# Cavity inner -Y wall at y = -62.5. Clearance: -59 - (-62.5) = 3.5 mm. Tight.
print(f"\nPi+Bonnet stack y extent ≈ [-59, -34]; cavity -Y wall at -62.5; clearance 3.5 mm.")
print("Tight but OK. But: lid screws (M3) need ~6 mm of thread length on the -Y wall.")


Confirmed: grommet hole is on the wrong side (extends -Y into the open bay
cavity instead of +Y into the chassis main body where wires would arrive).
And the +X wire slot inside the main chassis can't actually reach the bay
on the -Y side without traversing the bore-side of the chassis.

Pi+Bonnet stack y extent ≈ [-59, -34]; cavity -Y wall at -62.5; clearance 3.5 mm.
Tight but OK. But: lid screws (M3) need ~6 mm of thread length on the -Y wall.


In [17]:
# A couple more checks then I'll write the answer.

# Servo packaging check vs solenoid pocket clash (risk #9):
# Servo pocket: x ±20.5, y ∈ [13.7, 30], z ∈ [3.4, 46.6]
# Solenoid pocket: x ∈ [-35, -25.2], y ∈ [-9.6, 9.6], z ∈ [18.9, 41.1]
# Different Y sides (+Y vs -Y), so no direct clash. But both occupy z range ~[18, 46]
# near the dispense end. The horn slot (cut through bore at servo pocket z=25 ±4)
# punches the bore at z ∈ [21, 29]. The solenoid slot pierces bore at z ∈ [27, 33].
# Both bore-piercings within the same axial region (z ≈ 20-33).
# Result: bore wall has TWO simultaneous breaches at z ≈ 25-30, on opposite sides.
# Auger flute moving powder through this region encounters jagged interruptions
# in the bore wall on both X faces → leakage / jamming risk near dispense end.
print("CLASH CHECK z near dispense end:")
print("  Horn-slot bore breach: z ∈ [21, 29]")
print("  Solenoid bore slot:    z ∈ [27, 33]")
print("  → Overlap z=[27, 29]: bore wall is breached on +Y AND -X simultaneously.")
print("  Combined with the dispense funnel cone in the auger (z=0..12), the auger's")
print("  metering action just above the funnel is happening in a region with two")
print("  major bore-wall openings. Powder will leak sideways into both cavities.")

# Auger fin is INSIDE the auger's own tube, but bore is breached at z=21-33.
# Even if architecture were right, breaches at z<35 (above the funnel exit at z=12)
# would let powder escape through the slots before reaching the dispense exit.


CLASH CHECK z near dispense end:
  Horn-slot bore breach: z ∈ [21, 29]
  Solenoid bore slot:    z ∈ [27, 33]
  → Overlap z=[27, 29]: bore wall is breached on +Y AND -X simultaneously.
  Combined with the dispense funnel cone in the auger (z=0..12), the auger's
  metering action just above the funnel is happening in a region with two
  major bore-wall openings. Powder will leak sideways into both cavities.


In [18]:
# Print orientation gotcha: design prints vertical with dispense end (z=0) on
# build plate. Chassis is 300 mm tall — taller than most consumer FDM printers
# (Prusa Mk4: 220 mm Z; Bambu X1: 256 mm; Voron 2.4 350: 350 mm OK).
# Layer adhesion stress is maximized on a tall slender print. PLA's interlayer
# strength is ~50% of bulk. ERM vibration will excite the chassis tube modes:
# first bending mode of a 300mm × 70mm slender bar (cantilevered? simply supported on
# bench?) — typically tens of Hz. ERM at 12k rpm = 200 Hz is well above that.
# But sub-harmonics + start/stop transients can still excite modes.

import math
# Chassis tube as cantilever from bench (rough): L=0.3 m, rectangular section ~70×60 mm
# Iyy = b*h^3/12 = 0.07 * 0.06^3 / 12 = 1.26e-6 m^4
# A = 0.07*0.06 = 4.2e-3 m^2
# But the bore + e-bay cavity remove material. Approximate effective EI is ~70% of solid.
L = 0.3
b = 0.07; h = 0.06
I = b*h**3/12
A = b*h
m_per_L = 1.24e-3 * A * 1e6 * 0.6   # density g/mm^3 * A in m^2 → not quite right; redo
# Use rho directly: 1240 kg/m^3
rho = 1240
A_eff = A * 0.6   # account for cavities + infill
EI = 3.5e9 * I * 0.7
mu = rho * A_eff
f1_chassis = (1.875**2/(2*math.pi)) * math.sqrt(EI/(mu*L**4))
print(f"Chassis cantilever first bending mode: ≈ {f1_chassis:.1f} Hz")
# If this lands near solenoid duty cycle frequency or ERM startup, could couple.
# But chassis is normally mounted bench-bottom (not cantilevered) so this is mostly
# academic.


Chassis cantilever first bending mode: ≈ 195.4 Hz


In [19]:
# That's near the ERM operating frequency (200-250 Hz for 12-15k rpm).
# If the chassis is mounted as a free-standing tower (not bolted to bench), the
# ERM could excite the first bending mode directly. This is a real concern only
# if the chassis is upright on a small base; for bench-mounted, much higher modes.

# Final tally — I have enough to write up. Let me also note the CAD-agent
# workflow lesson (risk #10).

# Summary of findings to report (ranked):
#
# P0 (will fail on first print/run):
# 1. AUGER ARCHITECTURE: rotating tube + helix inside same body co-rotate with the
#    housing → no metering possible. The auger needs to be a SHAFT WITH FIN
#    (Ø<bore) inside the STATIONARY chassis bore, not a rotating tube+fin assembly.
# 2. AUGER COUPLER GAP: auger top at z=250, coupler chamber at z=274-300; auger
#    script does not extrude any M3 stub spanning that 24 mm gap. Can't be coupled.
# 3. AUGER FIN IS A COMPOUND, NOT A FUSE: shipped as cq.Compound. Slicer may
#    treat fin and tube body as overlapping separate solids with z-fighting,
#    OR the OCCT compound may export as a single mesh with a non-manifold
#    intersection. Print will either fail or produce a non-functional fin.
# 4. SOLENOID POCKET ORIENTATION: 22 mm dimension goes along Z, but the plunger
#    axis is along the 22 mm dimension — plunger ends up pointing vertically,
#    not at the bore.
# 5. E-BAY LID HAS NO THREAD ENGAGEMENT: lid opening removes the entire -Y wall;
#    the 4 corner heat-set inserts are drilled into empty space.
# 6. STEPPER-WIRE GROMMET IS DISCONNECTED: 3.5 mm deep cut in -Y face, but bay
#    cavity is 59 mm away. Wires can't reach.
# 7. SERVO PIVOT CLAIM IS NON-PHYSICAL: chassis is one solid block; there is no
#    hinged dispense nozzle for the servo to actuate. The horn slot just lets the
#    horn touch the auger flute. This rotates the auger, fights the stepper, or
#    jams.
#
# P1 (architectural / will work poorly):
# 8. ERM ENCAPSULATED IN PLA, 19 mm from bore — calculated coupling is ~1.5 mG at
#    the bore. Need an airgap or potting pocket near the bore.
# 9. ERM POCKET BRIDGES: pause Z=125 is at pocket MIDDLE; upper 5 mm of pocket
#    needs to be bridged after resume → sag risk. Pause should be at pocket top
#    (z=130.15).
# 10. SOLENOID THERMAL: continuous duty would melt PLA. No software duty-cycle
#    limit or thermal cutout in the design. Document and add.
# 11. AUGER CANTILEVER WHIP: marginal (whirl ~1350 rpm). Add a lower bushing
#    integrated in the chassis bore at z≈10 (just above the funnel) and a top
#    self-aligning bushing at z≈245.
# 12. BORE BREACHES OVERLAP at z=27–29 (horn slot + solenoid slot): two
#    simultaneous bore-wall openings 5–10 mm above the funnel exit. Powder will
#    leak.
# 13. CADSmith failure mode (risk #10): the 6-iteration manifold failure on the
#    helix is a structural workflow signal. CadQuery sweep+isFrenet on tight
#    helices is known-fragile; the agent should be biased toward (a) loft of
#    cross-sections at sampled helix points, or (b) construction in OpenSCAD,
#    or (c) bake the auger as a separately-parameterised SCAD/STL artifact
#    rather than insisting on cq.union into the chassis.
#
# P2 (nice-to-have / future):
# 14. Bonnet/Tic500 I2C-vs-USB clarification (Tic 500 must use USB if Bonnet is
#     on GPIO).
# 15. Powder spec missing — hygroscopic powders, particle size, and density all
#     drive pitch/diameter design.
# 16. 300 mm-tall print exceeds many consumer FDM Z-heights; specify printer.
# 17. Embedding electronics during print = unrepairable failure mode. ERM and
#     solenoid should be top-loadable in a side-cover that bolts on, not
#     entombed by print-resume.
print("Findings catalogued.")


Findings catalogued.
